# Lecture: Serving a Language Model (batching, streaming, queues)

Read this before opening notebook.ipynb. Each vocabulary term below gets its own section
(marked §) with a picture or worked numbers — skim the index, study the ones that feel fuzzy.

## Learning goals

- Explain endpoints and their contracts in one sentence each: 400 for bad input, 429 for overload, clamps for greedy inputs (§4).
- Explain why batching cuts per-prompt cost, with the B=1→8 sweep memorized (§5).
- Explain streaming, TTFT, and TPOT: first-word wait vs gaps between words (§6).
- Explain bounded queues and read one request's full timings end to end (§7–§8).


## The big idea

A trained model is a brilliant but slow chef; the server is the restaurant around them. Customers (**requests**) arrive at named counters (**endpoints** like `/generate`); the kitchen has only 4 stoves (**in-flight slots**), and extra crowds get a polite “we're full” card (**error 429**) instead of crashing the kitchen. Cooking one meal at a time wastes heat; cooking 8 similar meals together shares the prep (**batching**: 14.1 → 5.7 ms per prompt), so each meal costs less. Some diners want courses served as ready (**streaming**, one token frame at a time) instead of waiting for the whole feast. Two clocks matter: **TTFT** (time to first token: seconds until the first word arrives) and **TPOT** (time per output token: seconds between following words). Analogy: a bus (batch) is slower to fill than a taxi (single) but far cheaper per rider — unless the queue overflows, when the bouncer (semaphore) turns people away.


In [ ]:
import numpy as np  # bring in numpy for the tiny timing arrays
import matplotlib.pyplot as plt  # bring in pyplot for the taxi-vs-bus bars
import random  # bring in stdlib random for the seed habit
random.seed(17)  # fix stdlib seed so jitter matches each run
np.random.seed(17)  # fix numpy seed so the lecture matches each run
single = 14.1  # ms per prompt cooking one at a time (lab B=1)
batched = 5.7  # ms per prompt sharing one batch of 8 (lab B=8)
print('single per-prompt ms:', single)  # show lonely cooking costs 14.1 ms
print('batched per-prompt ms:', batched)  # show shared cooking costs 5.7 ms
print('speedup:', round(single / batched, 2))  # show ~2.47x cheaper per meal
y = np.arange(4)  # four toy diners standing in line
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))  # make two side-by-side panels
ax[0].barh(y, [single] * 4, color='tab:red')  # long red bars are lonely cooking
ax[0].set_title('One taxi each: slow')  # title in plain words
ax[0].set_xlabel('ms per prompt (shorter is faster)')  # label defines cost direction
ax[1].barh(y, [batched] * 4, color='tab:green')  # short green bars are the shared bus
ax[1].set_title('One bus of 8: fast')  # title in plain words
ax[1].set_xlabel('ms per prompt (shorter is faster)')  # label defines cost direction
fig.tight_layout()  # stop titles and labels from overlapping
plt.show()  # display the figure inside the notebook
print('Lesson: shared prep splits the bill across riders.')  # one-sentence takeaway
print('RECALIBRATE: constants above are the author CPU run (SEED=17); paste your results/batch_sweep.csv row to replace them - ordering/shape is the claim, digits are not')


## Vocabulary, one section each

Quick map — jump to whatever feels fuzzy. §4 guards the door, §5–§6 buy speed and feel, §7 buys survival.

| Term | One-liner | Section |
|---|---|---|
| Endpoint | A named counter (`/health`, `/generate`, `/batch`, `/metrics`) | §4 |
| 400 / 429 | Client error (empty prompt) / server-busy shed (in-flight limit hit) | §4 |
| Clamp | Cap wild inputs (`max_tokens=1000` → 64) so one diner cannot eat all day | §4 |
| Batching | Cook several prompts in one shared pass to split the fixed prep cost | §5 |
| Streaming (SSE) | Serve each word as ready (one frame per token) plus a DONE card | §6 |
| TTFT / TPOT | Wait for the first word / average gap between following words | §6 |
| Bounded queue | Fixed waiting room; bursts beyond it are shed (429), never queued forever | §7 |


### §4 Endpoints and contracts — the shapes table

**In one sentence: an endpoint is a named counter with a fixed request/response shape, and its contract says which bad inputs get 400 (your fault), which overloads get 429 (busy, retry later), and which wild inputs get clamped — never a crash.**

| Situation | Shape | Meaning, in plain words |
|---|---|---|
| Empty prompt | `400 'empty prompt'` | Client error: nothing to cook, rejected before the model |
| `max_tokens=1000` | clamped to `64` | Greedy input capped: one diner cannot eat all day |
| 5th concurrent request (4 stoves) | `429 'server busy'` | Shed fast: retry later beats queuing forever or crashing |
| Normal request | `200 {'text', 'tokens', 'lat_ms'}` | Success: text plus its timings |

Watch it: a plain-loop simulation of 8 requests against 4 stoves — 4 admitted (200), 2 shed (429), 1 empty rejected (400), 1 greedy clamped to 64.


In [ ]:
import numpy as np  # reuse numpy for the outcome tally
import matplotlib.pyplot as plt  # reuse pyplot for the outcome bars
np.random.seed(17)  # fix the seed for reproducibility
slots = 4  # kitchen stoves: at most 4 requests cooking at once
MAX_TOKENS = 64  # per-request cap: bounds worst-case decode latency
busy = 0  # stoves currently occupied
admitted, shed, bad = 0, 0, 0  # outcome counters: 200s, 429s, 400s
for i in range(6):  # six diners arrive at once (burst against 4 stoves)
    if busy < slots:  # a stove is free: admit
        busy += 1  # occupy one stove
        admitted += 1  # count the 200
    else:  # kitchen full: shed fast
        shed += 1  # count the 429 (busy, retry later)
print('admitted (200):', admitted)  # print 4: four stoves, four meals
print('shed (429):', shed)  # print 2: the bouncer turns two away
prompt = ''  # seventh diner orders nothing
if not prompt:  # empty prompt is a client error, never a model call
    bad += 1  # count the 400
print('rejected (400):', bad)  # print 1: bad input, instant refusal
asked, got = 1000, min(1000, MAX_TOKENS)  # eighth diner orders 1000 tokens
print('clamped:', asked, '->', got)  # print 1000 -> 64: greed capped
fig, ax = plt.subplots(figsize=(6, 3.5))  # make one figure for the outcomes
ax.bar(['200 ok', '429 busy', '400 bad'], [admitted, shed, bad], color=['tab:green', 'tab:orange', 'tab:red'])  # outcome bars
ax.set_ylabel('requests (of 7)')  # label the count axis
ax.set_title('Contracts: 4 served, 2 shed, 1 refused')  # title in printed numbers
plt.tight_layout()  # fix any label overlap before drawing
plt.show()  # render the bars

### §5 Batching — one shared prep, cheaper seats

**In one sentence: batching runs several prompts through one shared prefill pass, so the fixed prep cost splits across riders — per-prompt cost falls 14.1 → 5.7 ms from B=1 to B=8 (≈2.47x).**

Every request pays two bills: **prefill** (read the whole prompt at once — parallel, fixed-ish cost) plus **decode** (one token at a time). Alone, each prompt pays full prefill; batched, 8 prompts share one prefill and each pays ~1/8. The sweep proves it: B=1/2/4/8 → per-prompt 14.1/11.0/8.4/5.7 ms while tok/s climbs 1135 → 2807. Watch it: the per-prompt timeline bars (lonely vs shared) plus the full sweep curves.


In [ ]:
import numpy as np  # reuse numpy for the sweep arrays
import matplotlib.pyplot as plt  # reuse pyplot for bars + sweep
np.random.seed(17)  # fix the seed for reproducibility
B = np.array([1, 2, 4, 8])  # batch widths tested in the lab sweep
per = np.array([14.1, 11.0, 8.4, 5.7])  # ms per prompt: sharing cuts cost
tps = np.round(16 / per * 1000, 0)  # tok/s at 16 tokens each: cost inverted
print('per-prompt ms:', per.tolist())  # show cost falling 14.1 -> 5.7
print('tok/s:', tps.tolist())  # show speed rising 1135 -> 2807
print('speedup B1->B8:', round(float(per[0] / per[-1]), 2))  # print 2.47: the sharing win
fig, ax = plt.subplots(1, 2, figsize=(10, 4))  # two panels: timelines + sweep
y = np.arange(4)  # four toy prompts
ax[0].barh(y + 0.2, [14.1] * 4, 0.4, color='tab:red', label='B=1 lonely')  # long bars: full prefill each
ax[0].barh(y - 0.2, [5.7] * 4, 0.4, color='tab:green', label='B=8 shared')  # short bars: prefill split 8 ways
ax[0].set_xlabel('ms per prompt (shorter is faster)')  # label defines cost direction
ax[0].set_title('Shared prep splits the bill (14.1 -> 5.7)')  # title in printed numbers
ax[0].legend(fontsize=8)  # show which color is which width
ax[1].plot(B, per, marker='o', label='ms/prompt')  # cost falling with width
ax[1].plot(B, tps / 200, marker='s', label='tok/s ÷ 200 (scaled)')  # speed rising (scaled to fit)
ax[1].set_xlabel('batch width B')  # label names the knob simply
ax[1].set_ylabel('ms/prompt and tok/s÷200 (see legend)')  # label explains the scaling
ax[1].set_title('Wider bus -> cheaper seats, more tok/s')  # title in plain words
ax[1].legend(fontsize=8)  # show which line is which
ax[1].grid(True, alpha=0.3)  # add faint grid to read values easily
plt.tight_layout()  # stop titles and labels from overlapping
plt.show()  # display the figure

### §6 Streaming, TTFT, TPOT — feel fast, measure honestly

**In one sentence: streaming sends each token the moment it is ready (one frame per token, DONE at the end), TTFT is the wait until the first frame, and TPOT is the average gap between following frames.**

Total latency hides two different pains: a 10 ms reply could be “first word at 2 ms, then a word per ms” (feels instant) or “silence for 9 ms, then everything” (feels broken). Streaming fixes the feeling without changing the total: the same 8 tokens arrive at 2.4, 3.5, …, 10.1 ms — TTFT 2.4 ms, TPOT 1.1 ms, full latency 10.1 ms, ≈792 tok/s. The lab's `/generate_stream` endpoint is exactly this: one SSE frame per token plus DONE. Watch it: the token-arrival timeline with TTFT and TPOT marked.


In [ ]:
import numpy as np  # reuse numpy for the arrival-time array
import matplotlib.pyplot as plt  # reuse pyplot for the arrival timeline
np.random.seed(17)  # fix the seed for reproducibility
ttft, tpot, k = 2.4, 1.1, 8  # first-word wait, gap between words, token count
arr = ttft + tpot * np.arange(k)  # arrival ms of each token frame
lat = round(float(arr[-1]), 1)  # full latency = last arrival: 10.1 ms
print('TTFT ms:', ttft)  # print 2.4: responsiveness
print('TPOT ms:', tpot)  # print 1.1: steady word rate
print('arrivals:', [round(float(x), 1) for x in arr])  # print every frame time
print('full latency ms:', lat)  # print 10.1: same total, better feel
print('tok/s:', round(k / (lat / 1e3), 0))  # print 792.0: throughput view
fig, ax = plt.subplots(figsize=(8, 3))  # wide flat figure for the timeline
ax.eventplot(arr, lineoffsets=1, linelengths=0.5)  # one tick per token frame
ax.scatter([ttft], [1], s=120, color='tab:green', zorder=3)  # highlight the first frame
ax.annotate('TTFT 2.4', xy=(ttft, 1), xytext=(ttft - 1.5, 1.35), fontsize=9, arrowprops=dict(arrowstyle='->'))  # mark the first wait
ax.annotate('TPOT 1.1 gaps', xy=(arr[3], 1), xytext=(5.5, 0.55), fontsize=9, arrowprops=dict(arrowstyle='->'))  # mark the steady gaps
ax.set_xlim(0, 11)  # fix range to fit all 8 frames
ax.set_ylim(0.3, 1.6)  # fix range around the tick row
ax.set_yticks([])  # hide y: time is the only axis
ax.set_xlabel('ms since request (streaming frames)')  # label the time axis
ax.set_title('Streaming: first word at 2.4 ms, then one per 1.1 ms')  # title in printed numbers
plt.tight_layout()  # fix any label overlap before drawing
plt.show()  # render the timeline

### §7 Queues and clamps — survive the burst

**In one sentence: a bounded queue admits only what fits (here 4 concurrent), sheds the rest with 429 instead of queuing forever, and clamps cap each request's size — together they turn overload from a crash into a retry.**

Unbounded queues are slow-motion crashes: every burst eventually eats all memory (OOM) or times out (500s). The lab's semaphore holds 4 in-flight slots; the 5th+ concurrent request gets 429 immediately and the client retries with backoff. The what-if ablation shows the knob plainly: 8 instant arrivals against bound 4 → 4 served, 4 shed; against bound 64 → 8 served, 0 shed (room absorbs it). And `max_tokens` clamps keep one greedy request from becoming a burst of one. Watch it: the admission diagram with both bounds printed.


In [ ]:
import matplotlib.pyplot as plt  # only pyplot + stdlib: admission is counting
import random  # bring in stdlib random for the seed habit
random.seed(17)  # fix the seed for reproducibility
arrivals = 8  # one instant burst of 8 requests
served_small, shed_small = 4, 4  # bound 4: room for half the burst
served_big, shed_big = 8, 0  # bound 64: room absorbs the burst
print('bound 4: served', served_small, 'shed', shed_small)  # print 4, 4: bouncer works
print('bound 64: served', served_big, 'shed', shed_big)  # print 8, 0: room absorbs
fig, ax = plt.subplots(figsize=(8, 3.5))  # wide figure for the admission diagram
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 10)  # fix placement range
ax.set_ylim(0, 4)  # fix placement range
ax.text(1.2, 2.0, 'burst\n8 arrivals', ha='center', va='center', fontsize=9,  # left: the burst
        bbox=dict(boxstyle='round', facecolor='wheat'))  # wheat = incoming load
ax.text(5, 2.8, 'slots x4\n(4 served)', ha='center', va='center', fontsize=9,  # middle: admitted
        bbox=dict(boxstyle='round', facecolor='lightgreen'))  # green = cooking
ax.text(5, 1.0, 'shed x4\n(429 retry)', ha='center', va='center', fontsize=9,  # middle-low: shed
        bbox=dict(boxstyle='round', facecolor='mistyrose'))  # pink = turned away
ax.text(8.5, 2.0, 'metrics\ncount all', ha='center', va='center', fontsize=9,  # right: observability
        bbox=dict(boxstyle='round', facecolor='lightblue'))  # blue = counted
ax.annotate('', xy=(4.0, 2.8), xytext=(2.2, 2.2), arrowprops=dict(arrowstyle='->'))  # burst -> slots
ax.annotate('', xy=(4.0, 1.0), xytext=(2.2, 1.8), arrowprops=dict(arrowstyle='->'))  # burst -> shed
ax.annotate('', xy=(7.7, 2.0), xytext=(5.9, 2.6), arrowprops=dict(arrowstyle='->'))  # slots -> metrics
ax.set_title('Bounded queue: 8 arrivals, 4 stoves -> 4 x 429 (bound 64: 0 shed)')  # title in printed numbers
plt.tight_layout()  # snug the layout
plt.show()  # render the diagram

## §8 How it all fits together — the system map

One request's journey touches every § term in order. It knocks on an **endpoint** and passes the **contracts** (§4: non-empty? clamped?); waits in the bounded **queue** for a stove (§7); **batches** through one shared prefill with its neighbors (§5); **streams** home token by token while the clocks record **TTFT/TPOT** (§6); and lands in **metrics** (counts, median latency, tok/s). The map below pins each term to its station; the trace times one request through all of them.


In [ ]:
import matplotlib.pyplot as plt  # only pyplot: this cell is a pure diagram
import random  # bring in stdlib random only to satisfy the seed habit (unused here)
random.seed(17)  # fix the seed for reproducibility
fig, ax = plt.subplots(figsize=(11, 3.5))  # wide figure for the request pipeline
ax.axis('off')  # hide axes: diagram, not data
ax.set_xlim(0, 15)  # fix placement range
ax.set_ylim(0, 4)  # fix placement range
stages = [('request\n§4 shapes', 1), ('queue\n§7 bound 4', 4), ('batch\n§5 shared', 7), ('stream\n§6 frames', 10), ('metrics\ncounts', 13)]  # five pipeline stations
for name, xc in stages:  # draw each station box
    ax.text(xc, 2.2, name, ha='center', va='center', fontsize=8,  # centered station label
            bbox=dict(boxstyle='round', facecolor='lightblue'))  # blue = pipeline
for xa, xb in [(1.9, 3.0), (4.9, 6.0), (7.9, 9.0), (10.9, 12.0)]:  # arrow gaps between stations
    ax.annotate('', xy=(xb, 2.2), xytext=(xa, 2.2),  # draw a forward arrow left -> right
                arrowprops=dict(arrowstyle='->'))  # plain arrowhead style
ax.text(7, 3.4, 'FLOW: contract -> queue -> batch -> stream (§4-§7)', ha='center', fontsize=9, fontweight='bold')  # phase banner
ax.text(7, 0.9, 'CLOCKS: TTFT 2.4 ms, TPOT 1.1 ms, latency 10.1 ms (§6)', ha='center', fontsize=9, fontweight='bold', color='darkred')  # clocks banner
ax.set_title('System map: request -> queue -> batch -> stream -> metrics')  # diagram title
plt.tight_layout()  # snug the layout
plt.show()  # render the map

In [ ]:
import numpy as np  # reuse numpy for the latency arithmetic
np.random.seed(17)  # fix the seed for reproducibility
prompt, k = 'the king', 8  # one request: 8-char prompt, 8 tokens out
queue, prefill = 0.6, 1.8  # ms waiting for a stove + ms reading the prompt
ttft = round(queue + prefill, 1)  # first-word wait: 2.4 ms (§6)
tpot = 1.1  # steady gap between following words (§6)
lat = round(ttft + (k - 1) * tpot, 1)  # full latency: 10.1 ms
tps = round(k / (lat / 1e3), 0)  # throughput view: 792 tok/s
print('prompt:', repr(prompt), '| tokens out:', k)  # show the request shape (§4)
print('queue 0.6 + prefill 1.8 = TTFT', ttft)  # print 2.4: responsiveness
print('TPOT', tpot, 'x 7 gaps -> latency', lat)  # print 10.1: full reply time
print('tok/s:', tps)  # print 792.0: speed view of the same reply

## Bridge to the lab

Open `notebook.ipynb` now. Its cells build the real restaurant this lecture only simulated with loops (no servers or ports below — the lab uses `TestClient`, still no ports):

1. **Cell 1 (setup)** — seeds plus `os.makedirs('./results')`. Same `SEED = 17` habit as this lecture.
2. **Cell 2 (data + model)** — the same tiny L2/d64 chef from P16, briefly trained — the kitchen our bus serves.
3. **Cell 3 (app)** — the FastAPI restaurant: `/health`, `/generate` (clamped, §4), `/generate_stream` (SSE frames + DONE, §6), `/batch` (one shared prefill, §5), semaphore of 4 with 429s (§7), `/metrics`.
4. **Cell 4 (tests + benches)** — checks contracts (§4: empty → 400, huge `max_tokens` → 64), then times singles vs batch-8 (§5) and 8 racers vs 4 stoves (§7) into `results/server.csv`.
5. **Cell 5 (sweeps + figure)** — sweeps B=1/2/4/8 (§5: per-prompt 14.1 → 5.7 ms) and queue bounds (§7) into `results/batch_sweep.csv` and `results/server.png`.
6. **Cell 6 (cleanup)** — serve for real with `uvicorn notebook:app`; admits greedy decode, no KV-cache, no continuous batching — P18's upgrades.


## Self-check — predict first, then check

> Retrieval rule: cover every *Answer* below, write your prediction (a number, a shape, or one sentence), run the § cell it cites, then uncover. The last question in each set has no printed answer — do it from the map.



1. **Batch-8 is 2.47x cheaper per prompt (5.7 vs 14.1 ms). Where did the savings come from?**

*Answer: the shared prefill (§5). Eight prompts split one fixed prompt-reading pass instead of paying it 8 times — decode still costs per token, which is why the win is ~2.5x and not 8x. Trace one request's bills in the §8 numbers.*

2. **8 racers hit 4 stoves and 4 get 429s. Bug or feature?**

*Answer: feature (§4 + §7). The semaphore sheds with 429 (busy, retry later) instead of queuing forever or crashing — bound 4 sheds 4 of 8 while bound 64 sheds 0. Backpressure is the product.*

3. **TTFT is 2.4 ms but full latency is 10.1 ms. Why report both?**

*Answer: they measure different pains (§6). TTFT (queue 0.6 + prefill 1.8) reports responsiveness — streaming shows it instantly; full latency adds 7 TPOT gaps of 1.1 ms. One clock cannot tell you the reply felt instant.*
